# ViSceT5 — PreSTU + Region Grounding (nhánh `exp/pretrain-grounding-avf`)Tiền huấn luyện cho **QA-CLIP** và **AVF** — hai module đang thấp hơn baseline trong bảng ablation.### Nhánh này sửa gì| Vấn đề đo được | Sửa ||---|---|| `patch_scores` của QA-CLIP **hằng số** (std giữa các patch = `2.3e-10`) ⇒ AVF cắt **toàn bộ ảnh** với mọi câu hỏi | Bản đồ grounding chuẩn hoá theo chiều **patch**; p99 giờ chọn 2/196 ô thay vì 196/196 || Adapter QA-CLIP nhận **0 gradient** khi `vision_unfreeze_last_n=0` (bị `no_grad` nuốt) | Cổng `no_grad` xét theo `requires_grad` thật của `qa_clip` || QA-CLIP/AVF chỉ học qua CE câu trả lời ⇒ gate ReZero không mở | **Loss grounding**: nhãn vùng 14×14 dạy trực tiếp "nhìn vào đâu", cho cả bản đồ AVF lẫn bản đồ QA-CLIP || `AVFFusion` học ở không gian T5 nhưng dùng ở không gian CLIP | Gộp chung một đường cho cả hai giai đoạn || Crop đọc bằng ConvNeXt-V2 (ImageNet, không đọc được chữ) | Crop đi qua **chính CLIP ViT** (AnyRes, dùng chung trọng số) — `VisualSearch: 27.87M → 0` tham số || 20.1M tham số OCR (`OCREncoder`+`SemanticOCREmbedding`+char) **không train** trong pretrain | Chạy SceSpaVis ở cả hai giai đoạn, pretrain chỉ nạp **từ prefix** (không rò rỉ target) || Target dài hàng chục từ ↔ finetune trả lời ngắn | `max_target_words=5` || Chế độ full-OCR dạy "nhìn khắp ảnh" | Loại khỏi loss grounding + bbox (giữ mục tiêu sinh chữ) |**Quay về bản cũ bất cứ lúc nào:** đổi `BRANCH` ở cell 2 thành `exp/pretrain-gen-all`.---### Cấu hình Kaggle trước khi chạy1. **Accelerator:** GPU T4 / P100 (T4 x2 cũng được)2. **Internet:** **ON** (bắt buộc — clone repo, tải weights)3. **Persistence:** Files + Variables

## 1. Clone repo & checkout nhánh

In [ ]:
import os, sysBRANCH   = "exp/pretrain-grounding-avf"     # đổi thành "exp/pretrain-gen-all" để quay về bản cũWORK_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "/content"REPO_DIR = os.path.join(WORK_DIR, "ViSceT5")if not os.path.exists(REPO_DIR):    !git clone https://github.com/Kussssssss/ViSceT5.git {REPO_DIR}%cd {REPO_DIR}!git fetch origin --quiet!git checkout {BRANCH}!git pull origin {BRANCH}!git log --oneline -4if REPO_DIR not in sys.path:    sys.path.insert(0, REPO_DIR)

## 2. Cài môi trường (transformers cố định 4.45.2)

In [ ]:
%%capture!pip uninstall -y transformers peft accelerate 2>/dev/null || true!pip install -q -r requirements.txt!pip install -q git+https://github.com/salaniz/pycocoevalcap!pip install -q --upgrade --no-cache-dir gdown

In [ ]:
import torch, transformersprint(f"PyTorch      : {torch.__version__}  (CUDA: {torch.cuda.is_available()})")print(f"Transformers : {transformers.__version__}  (cần 4.45.x)")if torch.cuda.is_available():    p = torch.cuda.get_device_properties(0)    print(f"GPU          : {p.name}  |  {p.total_memory/1024**3:.1f} GB  |  x{torch.cuda.device_count()}")assert transformers.__version__.startswith("4.45"), f"Cần transformers 4.45.x, đang có {transformers.__version__}"

## 3. Chuẩn bị dữ liệu (VinText + EVJVQA)

In [ ]:
%env OUTPUT_PATH=./output/pretrain!python scripts/prepare_dataset.py --config configs/data/VinText.yaml,configs/data/EVJVQA.yaml

## 4. Khởi tạo trọng số nền (ViT5 base + CLIP ViT-B/16)

In [ ]:
!python scripts/init_model.py

## 5. Kiểm tra nhanh trước khi tốn GPUCell này xác nhận 3 điều **trước** khi chạy full 10 epoch — nếu một trong ba sai thì chạy tiếp là phí giờ GPU:1. bản đồ grounding của QA-CLIP không còn hằng số (bug cũ khiến AVF cắt cả ảnh);2. nhãn grounding 14×14 hợp lệ và **không có từ target nào rò rỉ** sang prompt/khối OCR;3. chế độ full-OCR được đánh sentinel để không kéo heatmap về đều.

In [ ]:
import torch, torch.nn.functional as Ffrom transformers import CLIPVisionConfigfrom models.modules.qa_clip import MMCLIPAttentiontorch.manual_seed(0)att = MMCLIPAttention(CLIPVisionConfig(hidden_size=768, num_attention_heads=12,                                       intermediate_size=3072)).eval()B, VIS, MM = 2, 197, 32km = torch.zeros(B, MM); km[:, :7] = 1with torch.no_grad():    _, attn = att(torch.randn(B, VIS, 768), kv_states=torch.randn(B, MM, 768),                  kv_masks=km, output_attentions=True)ps = attn[:, :, 1:, :].mean(dim=[1, 3])g  = ps.reshape(B, 14, 14); f = g.reshape(B, -1)gn = (g - f.min(1, True)[0].unsqueeze(-1)) / (f.max(1, True)[0].unsqueeze(-1) - f.min(1, True)[0].unsqueeze(-1) + 1e-8)thr = torch.topk(gn.reshape(B, -1), 2, dim=1)[0][:, -1]n_cell = (gn >= thr.view(B, 1, 1)).sum(dim=(1, 2)).tolist()print(f"[1] std giữa các patch = {float(ps.std(dim=1).mean()):.3e}   (bug cũ: ~2.3e-10)")print(f"    số ô qua ngưỡng p99 = {n_cell} / 196   (bug cũ: 196/196 = cắt cả ảnh)")assert float(ps.std(dim=1).mean()) > 1e-8, "Bản đồ grounding vẫn hằng số!"assert max(n_cell) < 50, "Ngưỡng p99 vẫn giữ gần hết ảnh!"print("    OK\n")from data.collator import _boxes_to_patch_mask, _union_box, _split_ocr_spatial_regionwords = ["pepsi", "cola", "quán", "ăn", "bún", "bò", "huế", "số", "12", "nguyễn", "trãi"]gxy   = torch.rand(len(words), 2) * 0.8boxes = torch.cat([gxy, gxy + 0.08], 1).clamp(0, 1)leak = n_full = 0for t in range(200):    pw, pb, pi, tw, tb, ti, is_full = _split_ocr_spatial_region(list(words), boxes.clone(),                                                                max_target_words=5)    leak   += len(set(tw) & set(pw))    n_full += int(is_full)    if not is_full:        assert len(tw) <= 5, f"target {len(tw)} từ, vượt cap 5"        m = _boxes_to_patch_mask((tb.float() / 1000.0).clamp(0, 1), 14)        assert abs(float(m.sum()) - 1.0) < 1e-5 and int((m > 0).sum()) < 196print(f"[2] số từ target rò rỉ sang prefix  = {leak}   (phải = 0)")print(f"    target luôn <= 5 từ ở chế độ cụm, nhãn 14x14 tổng = 1.0 và không phủ hết")print(f"[3] chế độ full-OCR gặp {n_full}/200 lần -> được đánh sentinel, loại khỏi grounding+bbox")assert leak == 0print("\nTẤT CẢ KIỂM TRA ĐỀU ĐẠT — chạy tiếp được.")

## 6. Smoke test (~3-5 phút)Chạy vài chục step để chắc chắn dữ liệu, VRAM và cả 3 loss đều ổn trước khi vào full run.

In [ ]:
%env STAGE=pretrain%env MOCK_TEST=true%env SMOKE_TRAIN_SAMPLES=64%env SMOKE_EVAL_SAMPLES=16%env SMOKE_MAX_STEPS=12%env VISION_UNFREEZE_LAST_N=4!python run_pipeline.py 2>&1 | tail -45

## 7. Tiền huấn luyện đầy đủ| Tham số | Giá trị | Vì sao ||---|---|---|| `num_train_epochs` | 10 | || batch 4 × grad-accum 4 | hiệu dụng 16 | vừa VRAM T4 sau khi bỏ ConvNeXt || `learning_rate` | 1e-4 (ViT5), 1e-5 (CLIP) | LR vi sai || `vision_unfreeze_last_n` | 4 | mở tầng 8–11 + post-LN || `lambda_bbox_ce` | 0.3 | định vị box hợp của cụm || **`lambda_ground`** | **0.5** | **loss grounding — thứ dạy QA-CLIP/AVF nhìn đúng chỗ** || `num_bbox_bins` | 200 | 1000 bin là quá mịn so với lưới 14×14 || `max_target_words` | 5 | target dạng câu trả lời ngắn, cụm chặt || `vs_crop_encoder` | `clip` | crop đi qua chính CLIP ViT |> Trên T4, 10 epoch mất khoảng 8–11 giờ. Kaggle giới hạn 12h/phiên — đặt `save_total_limit` và> `PRETRAIN_HF_REPO` để đẩy checkpoint lên HF, phiên sau resume tiếp được.

In [ ]:
import osos.environ.pop("MOCK_TEST", None)# Đẩy checkpoint lên HF để phiên sau resume (bỏ trống nếu không dùng)# %env HF_TOKEN=hf_xxxxxxxx# %env PRETRAIN_HF_REPO=Kus669/ViSceT5-pretrain-grounding!python training/pretrain.py configs/pretrain.yaml \    --dataset_name "VinText,EVJVQA" \    --num_train_epochs 10 \    --per_device_train_batch_size 4 \    --gradient_accumulation_steps 4 \    --learning_rate 0.0001 \    --vision_unfreeze_last_n 4 \    --lambda_bbox_ce 0.3 \    --lambda_ground 0.5 \    --num_bbox_bins 200 \    --max_target_words 5 \    --vs_crop_encoder clip \    --save_total_limit 1 \    --output_dir /kaggle/working/pretrain_output \    --logging_dir /kaggle/working/pretrain_output/logs \    2>&1 | tee /kaggle/working/pretrain.log

## 8. Đọc log: cái gì báo hiệu thành công / thất bại| Chỉ số | Kỳ vọng | Nếu sai ||---|---|---|| `loss_ground` | giảm rõ từ **~5.3** (= ln 196). Tối ưu lý thuyết ≈ entropy của nhãn (~1.5) | **Phẳng ở 5.3** ⇒ grounding không chạy: kiểm tra `target_patch_mask` có trong batch không || `loss_text` | giảm đều | || `loss_bbox` | giảm từ ~6 | phẳng ⇒ xem `target_bbox_bins` có toàn `-100` không || `tanh(avf_fusion.gate)` | rời khỏi 0 sau 2–3 epoch | vẫn ≈0 ⇒ crop chưa giúp gì, xem lại cell 9 || `tanh(instruction_proj_gate)` | rời khỏi 0 | vẫn ≈0 ⇒ QA-CLIP vẫn đứng im |Hai gate là thước đo trực tiếp cho câu hỏi "QA-CLIP và AVF có thực sự hoạt động không".

In [ ]:
# Theo dõi hai cổng ReZero — chúng cho biết QA-CLIP và AVF có thật sự bật lên hay khôngimport glob, torch, osfrom safetensors.torch import load_fileck = sorted(glob.glob("/kaggle/working/pretrain_output/checkpoint-*"),            key=lambda p: int(p.rsplit("-", 1)[1]))assert ck, "Chưa có checkpoint nào."path = os.path.join(ck[-1], "model.safetensors")sd = load_file(path) if os.path.exists(path) else torch.load(    os.path.join(ck[-1], "pytorch_model.bin"), map_location="cpu")print(f"Checkpoint: {ck[-1]}\n")avf = [(k, float(torch.tanh(v.float().flatten()[0]))) for k, v in sd.items()       if k.startswith("avf_fusion") and "gate" in k]qav = [(k, float(torch.tanh(v.float().flatten()[0]))) for k, v in sd.items()       if "instruction_proj_gate" in k]print("AVF ReZero gate:")for k, v in avf:    print(f"   {v:+.5f}   {k}")print("\nQA-CLIP instruction gate (mỗi tầng):")for k, v in sorted(qav):    print(f"   {v:+.5f}   {k}")if qav:    mx = max(abs(v) for _, v in qav)    print(f"\n|gate| lớn nhất của QA-CLIP = {mx:.5f}")    print("   -> " + ("ĐÃ MỞ, module đang đóng góp thật." if mx > 0.01 else                      "vẫn đóng: QA-CLIP chưa tìm được lợi ích — tăng lambda_ground hoặc VISION_LR_SCALE."))

## 9. Trực quan hoá: bbox dự đoán + heatmap sau khi được dạy grounding

In [ ]:
from scripts.visualize_pretrain import visualize_pretrain_samplesfigs = visualize_pretrain_samples(    checkpoint="/kaggle/working/pretrain_output",    val_csv=None,    sample_idx=0,    num_samples=5,    save_dir="/kaggle/working/pretrain_output/visualizations",    show_plot=True,)print(f"\nĐã dựng {len(figs)} mẫu. Heatmap giờ phải BÁM vào cụm chữ target, "      f"không còn trải đều cả ảnh như trước.")

## 10. Memorization: model đang ĐỌC ảnh hay chỉ NHỚ ảnh?**Chữ ký OCR** = tập các từ OCR đọc được trên một ảnh. Nếu chữ ký duy nhất trong corpus,chỉ cần nhận ra vài từ là biết đang xem ảnh nào, rồi đọc nốt phần còn lại **từ trí nhớ**.Pretext "sinh các từ OCR không nằm trong prefix" **giải trọn vẹn được bằng trí nhớ**.Đo trên **chính corpus pretrain** (VinText 1.985 + EVJVQA 3.503 = 5.488 ảnh, không cần model nào):| Số đo | Giá trị ||---|---|| Ảnh có chữ ký OCR **duy nhất** | **99,1%** (5.437/5.488) || Riêng chuỗi prefix đã định danh duy nhất 1 ảnh | **81,5%** (VinText 88,7% · EVJVQA 77,2%) || — prefix 1–2 từ | 34,7% || — prefix 3–5 từ | **91,4%** || — prefix 6–10 từ | **97,5%** || — prefix >10 từ | **99,7%** || Số từ OCR trung bình mỗi ảnh | 20,1 || Tổng văn bản cần nhớ để giải 100% | **~401 KB** || Tham số khả huấn luyện | 291M ≈ 554 MB (bf16) — **dư ~1.400 lần** |(Tính bằng ngữ nghĩa tập hợp, bỏ qua số lần lặp của từ ⇒ các tỉ lệ trên là **chặn dưới**.)Nghĩa là: `loss_text` giảm **không** chứng minh model biết đọc. Nó có thể chỉ đang trabảng "ảnh này → chuỗi OCR này" rồi trừ đi phần prefix. PreSTU gốc tránh được vì dùng12M+ ảnh; ở quy mô này thì không.Cell dưới đo 4 điều kiện. Phép làm mờ phá nét chữ nhưng **giữ bố cục/màu**, tức giữnguyên "dấu vân" để nhận dạng ảnh và chỉ lấy đi khả năng đọc — nhờ vậy tách được haigiả thuyết mà một mình khoảng cách train/val không tách nổi.

In [ ]:
!python scripts/probe_memorization.py     --checkpoint /kaggle/working/pretrain_output     --data_dir ./output/pretrain     --num_samples 200     --blur_radius 4.0

Đọc kết quả:| Quan sát | Nghĩa là | Việc cần làm ||---|---|---|| `train − val` > 15 điểm | có memorization | dừng sớm theo **F1 trên val**, đừng theo loss train || làm mờ ảnh train mà điểm **không** giảm | đang NHỚ chứ không ĐỌC | tăng augmentation ảnh, giảm epoch || làm mờ ảnh val mà điểm **sập** | đang đọc thật trên ảnh chưa thấy | đúng thứ ta muốn |**Cột `point` mới là thước đo quyết định cho nhánh này.** Nó đo argmax của bản đồ liên quancó rơi đúng vùng target không (ngẫu nhiên ≈ 4%). Với mục tiêu "làm QA-CLIP và AVF vượtbaseline", cái cần tổng quát hoá là **grounding**, không phải khả năng đọc:* `point` trên val < 25% → bản đồ gần như chỉ bừa trên ảnh mới ⇒ QA-CLIP/AVF **sẽ không**  khá lên ở finetune, dù `loss_ground` lúc train có đẹp.* `point` train ≫ val → bản đồ bị nhớ theo ảnh ⇒ dừng sớm theo chính `point` trên val.Nếu rơi vào các trường hợp xấu thì đừng tin bảng ablation — hãy giảm số epoch tới trướcđiểm `point`-val bắt đầu đi ngang.

## 11. Chuyển giao sang finetune ViTextVQAKhác với trước, lần này chuyển giao được **toàn bộ** đường xử lý:`vit5`, `qa_clip` (kể cả adapter chỉ dẫn đã học grounding), `avf_fusion` (giờ cùng không gianbiểu diễn với finetune), và **20.1M tham số SceSpaVis** đã thực sự được huấn luyện.Chạy ablation để so với baseline: bật/tắt qua `ABLATION_USE_QACLIP` / `ABLATION_USE_VS` / `ABLATION_USE_OCR`.

In [ ]:
%env STAGE=finetune%env PRETRAIN_HF_REPO=Kus669/ViSceT5-pretrain-grounding# hoặc dùng checkpoint local:# %env MODEL_NAME_OR_PATH=/kaggle/working/pretrain_output/checkpoint-XXXX%env ABLATION_USE_QACLIP=true%env ABLATION_USE_VS=true%env ABLATION_USE_OCR=true%env OUTPUT_DIR=/kaggle/temp/finetune_grounding!python run_pipeline.py 2>&1 | tee /kaggle/working/finetune.log